In [88]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [89]:
df=pd.read_csv('loan_data.csv')

In [90]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [91]:
x=df.drop(columns='loan_status')
y=df['loan_status']

In [92]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [93]:
obj_cols=x.select_dtypes(include='object').columns

In [94]:
x_train[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [95]:
df['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [96]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [107]:
ordinal_encoder=[['High School','Bachelor','Master','Associate','Doctorate']]

onehot_encoder=['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']

In [108]:
pre_processing=ColumnTransformer(
    transformers=[
        ('onehotencoder',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),onehot_encoder),
        ('ordinalencoder',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1,categories=ordinal_encoder),['person_education'])
    ],remainder='passthrough'
)

In [109]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',pre_processing),
        ('model',DecisionTreeClassifier(random_state=42))
    ]
)

In [110]:
grid_search_cv=GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion':['gini','entropy'],
        'model__splitter':['best','random'],
        'model__max_depth':[None,5,10],
        'model__min_samples_split':[2,5],
        'model__min_samples_leaf':[1,3,5]
    },
    verbose=10, # verbose tells Scikit-learn how much information to print while the model is running
    n_jobs=-1   # n_jobs tells Scikit-learn how many CPU cores to use
)

In [111]:
grid_search_cv.fit(x_train,y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 5], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,10
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehotencoder', ...), ('ordinalencoder', ...)]"


In [112]:
grid_search_cv.best_estimator_

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...), ('ordinalencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [113]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [117]:
results=pd.DataFrame(grid_search_cv.cv_results_)

# This is a dictionary containing the performance of every parameter combination.

In [115]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.407285,0.032887,0.042455,0.004234,gini,None,1,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.893194,0.899444,0.894306,0.901111,0.897639,0.897139,0.002998,52
1,0.289572,0.015454,0.044533,0.004370,gini,None,1,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.887917,0.886528,0.877083,0.888194,0.886944,0.885333,0.004170,60
2,0.434996,0.023310,0.048327,0.011116,gini,None,1,5,best,"{'model__criterion': 'gini', 'model__max_depth...",0.897778,0.900972,0.898333,0.905556,0.901806,0.900889,0.002787,49
3,0.291249,0.013427,0.046597,0.004128,gini,None,1,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.890278,0.893611,0.892361,0.888056,0.893056,0.891472,0.002048,58
4,0.474075,0.041074,0.043610,0.003333,gini,None,3,2,best,"{'model__criterion': 'gini', 'model__max_depth...",0.899028,0.902500,0.904583,0.903056,0.898889,0.901611,0.002271,46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,0.295733,0.020432,0.044012,0.009086,entropy,10,3,5,random,"{'model__criterion': 'entropy', 'model__max_de...",0.899306,0.911389,0.905139,0.912361,0.903333,0.906306,0.004934,31
68,0.448975,0.027930,0.040735,0.002506,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
69,0.322465,0.026850,0.043096,0.004905,entropy,10,5,2,random,"{'model__criterion': 'entropy', 'model__max_de...",0.908889,0.911250,0.903472,0.908472,0.910278,0.908472,0.002689,27
70,0.421579,0.025616,0.029997,0.006438,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2


In [119]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
60,0.401264,0.025096,0.048114,0.013078,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918889,0.925972,0.914861,0.921111,0.920833,0.920333,0.003597,1
70,0.421579,0.025616,0.029997,0.006438,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
68,0.448975,0.027930,0.040735,0.002506,entropy,10,5,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
62,0.494652,0.042400,0.047657,0.007991,entropy,10,1,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919028,0.925972,0.915000,0.920694,0.920694,0.920278,0.003527,2
66,0.475789,0.058395,0.043624,0.004195,entropy,10,3,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918472,0.926111,0.915417,0.920000,0.920000,0.920000,0.003484,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,0.378854,0.063182,0.043974,0.004702,gini,5,3,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
19,0.279992,0.007826,0.037398,0.002837,gini,5,3,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
21,0.255657,0.006444,0.039962,0.005992,gini,5,5,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
23,0.240722,0.011770,0.038235,0.005097,gini,5,5,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872361,0.871667,0.872500,0.872639,0.872444,0.000453,67
